# Notebook 04: Ian no-trade-band threshold prototype

This notebook is a Colab-ready, `runner_utils`-orchestrated research prototype for Ian's no-trade-band direction. It keeps the first rung narrow: diagnostics and optional LSTM-only smoke by default, with TCN/DLinear available only as explicitly selected existing `runner_utils` baselines.

The stock-aware multi-scale DLinear + residual TCN idea is recorded as a proposed architecture spec, but it is not trained here. That model should move through a separate `runner_utils` implementation/review path before experiment comparison.

In [ ]:
import copy
import json
import os
import random
import sys
import time
from datetime import time as dtime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / "runner_utils").exists() and str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if (PROJECT_ROOT.parent / "runner_utils").exists() and str(PROJECT_ROOT.parent) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT.parent))

from runner_utils.config import DataConfig, TrainConfig, WindowConfig
from runner_utils.dataset import WindowedClassificationDataset
from runner_utils.dataset import fit_scaler_on_train
from runner_utils.dataset import make_no_trade_band_labels
from runner_utils.dataset import make_time_splits
from runner_utils.dataset import transform_split
from runner_utils.dataset import trim_labels_at_split_boundary
from runner_utils.metrics import always_predict_baseline_metrics
from runner_utils.metrics import compute_classification_metrics
from runner_utils.metrics import dummy_baseline_metrics
from runner_utils.models.dlinear_classifier import DLinearClassifier
from runner_utils.models.lstm_classifier import LSTMClassifier
from runner_utils.models.tcn_classifier import TCNClassifier
from runner_utils.seed import seed_everything
from runner_utils.trainer import evaluate, train_one_epoch

FULL_RUN = False
RUN_DATA_DIAGNOSTICS = False
RUN_TRAINING = False
WRITE_ARTIFACTS = False
ALLOW_OVERWRITE = False

DATA_ROOT = "/content/drive/MyDrive/stockdata/Dow_30_1min/"
TICKERS = ["CSCO", "JPM", "KO", "MSFT", "WMT"]
SELECTED_TICKERS = ["CSCO"]

BAR_FREQ = "5min"
WINDOW_SIZE = 12
LABEL_HORIZON_K = 12
DIAGNOSTIC_THRESHOLD_BPS_LIST = [0, 3, 5, 10]
MAIN_THRESHOLD_BPS_LIST = [5]
SENSITIVITY_THRESHOLD_BPS_LIST = [10]
SELECTED_THRESHOLD_BPS_LIST = [5]

AVAILABLE_BASELINE_MODELS = ["lstm", "tcn", "dlinear"]
SELECTED_MODELS = ["lstm"]
SELECTED_SEEDS = [42]

MAX_RAW_ROWS_PER_TICKER = 20000
MAX_EPOCHS = 1
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
STRIDE = 1

MARKET_OPEN = dtime(9, 30)
MARKET_CLOSE = dtime(16, 0)
TIMESTAMP_COL = "timestamp"
TICKER_COL = "ticker"
PRICE_COL = "close"
LABEL_COL = "label"
RAW_COLUMNS = ["date", "time", "open", "high", "low", "close", "volume"]
OHLCV_COLS = ["open", "high", "low", "close", "volume"]
FEATURE_COLS = ["open", "high", "low", "close", "volume", "macd", "macd_signal", "macd_hist", "rsi_14", "bb_pct_b", "rolling_std_20", "obv_roc"]
ARTIFACT_ROOT = "/content/drive/MyDrive/stockdata/notebook04_ian_threshold_outputs"

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
device = "cuda" if torch.cuda.is_available() else "cpu"

runtime_info_df = pd.DataFrame([{"python": sys.version.split()[0], "torch": torch.__version__, "cuda_available": torch.cuda.is_available(), "device": device, "full_run": FULL_RUN, "run_data_diagnostics": RUN_DATA_DIAGNOSTICS, "run_training": RUN_TRAINING, "write_artifacts": WRITE_ARTIFACTS, "allow_overwrite": ALLOW_OVERWRITE}])
display(runtime_info_df)

In [ ]:
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB and (RUN_DATA_DIAGNOSTICS or RUN_TRAINING):
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Drive mount skipped. Set RUN_DATA_DIAGNOSTICS=True or RUN_TRAINING=True in Colab to touch Drive data.")

threshold_scope_df = pd.DataFrame([
    {"threshold_bps": 0, "role": "no-band reference only", "default_training": False},
    {"threshold_bps": 3, "role": "low-band diagnostic", "default_training": False},
    {"threshold_bps": 5, "role": "main Phase 1B candidate", "default_training": True},
    {"threshold_bps": 10, "role": "sensitivity only", "default_training": False},
])

planned_grid_df = pd.DataFrame([{"threshold_bps": threshold_bps, "model_name": model_name, "ticker": ticker, "seed": seed, "scope": "staged_runner_utils_orchestration"} for threshold_bps in SELECTED_THRESHOLD_BPS_LIST for model_name in SELECTED_MODELS for ticker in SELECTED_TICKERS for seed in SELECTED_SEEDS])

display(threshold_scope_df)
display(planned_grid_df)

## Data Loading And Feature Preparation

Raw loading, 5-minute resampling, and technical-indicator preparation stay in this notebook because `runner_utils` expects prepared OHLCV plus feature columns. All label, split, scaler, window, metric, baseline, model, and training paths below call `runner_utils`.

In [ ]:
def validate_price_frame(df, ticker, context):
    missing = [col for col in [TIMESTAMP_COL, *OHLCV_COLS] if col not in df.columns]
    if missing:
        raise ValueError(f"{context}: {ticker} missing columns {missing}")
    if not pd.api.types.is_datetime64_any_dtype(df[TIMESTAMP_COL]):
        raise ValueError(f"{context}: {ticker} timestamp must be datetime dtype")
    if not df[TIMESTAMP_COL].is_monotonic_increasing:
        raise ValueError(f"{context}: {ticker} timestamp must be strictly increasing")
    if df[TIMESTAMP_COL].duplicated().any():
        raise ValueError(f"{context}: {ticker} timestamp has duplicates")
    if df[OHLCV_COLS].isna().any().any():
        raise ValueError(f"{context}: {ticker} OHLCV columns contain NaN")
    if (df[PRICE_COL] <= 0).any():
        raise ValueError(f"{context}: {ticker} close must be positive")
    if (df["volume"] < 0).any():
        raise ValueError(f"{context}: {ticker} volume must be non-negative")

def read_one_minute_ticker(ticker):
    path = Path(DATA_ROOT) / f"{ticker}.txt"
    if not path.exists():
        raise FileNotFoundError(f"Missing raw data file: {path}")
    nrows = None if MAX_RAW_ROWS_PER_TICKER is None else int(MAX_RAW_ROWS_PER_TICKER)
    df = pd.read_csv(path, header=None, names=RAW_COLUMNS, nrows=nrows)
    df[TIMESTAMP_COL] = pd.to_datetime(df["date"] + " " + df["time"], format="%m/%d/%Y %H:%M")
    df = df.drop(columns=["date", "time"]).sort_values(TIMESTAMP_COL).reset_index(drop=True)
    intraday = df[TIMESTAMP_COL].dt.time
    df = df[(intraday >= MARKET_OPEN) & (intraday <= MARKET_CLOSE)].reset_index(drop=True)
    validate_price_frame(df, ticker, "read_one_minute_ticker")
    return df

def resample_to_bars(df, ticker):
    bars = df.set_index(TIMESTAMP_COL).resample(BAR_FREQ).agg({"open": "first", "high": "max", "low": "min", "close": "last", "volume": "sum"}).dropna(subset=["open", "high", "low", "close"]).reset_index()
    intraday = bars[TIMESTAMP_COL].dt.time
    bars = bars[(intraday >= MARKET_OPEN) & (intraday <= MARKET_CLOSE)].reset_index(drop=True)
    bars[TICKER_COL] = ticker
    validate_price_frame(bars, ticker, "resample_to_bars")
    return bars

def compute_technical_indicators(df, ticker):
    out = df.copy(deep=True)
    ema12 = out[PRICE_COL].ewm(span=12, adjust=False).mean()
    ema26 = out[PRICE_COL].ewm(span=26, adjust=False).mean()
    out["macd"] = ema12 - ema26
    out["macd_signal"] = out["macd"].ewm(span=9, adjust=False).mean()
    out["macd_hist"] = out["macd"] - out["macd_signal"]
    delta = out[PRICE_COL].diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)
    avg_gain = gain.ewm(span=14, adjust=False).mean()
    avg_loss = loss.ewm(span=14, adjust=False).mean()
    rs = avg_gain / avg_loss
    out["rsi_14"] = 100.0 - (100.0 / (1.0 + rs))
    sma20 = out[PRICE_COL].rolling(20).mean()
    std20 = out[PRICE_COL].rolling(20).std()
    upper = sma20 + 2.0 * std20
    lower = sma20 - 2.0 * std20
    out["bb_pct_b"] = (out[PRICE_COL] - lower) / (upper - lower)
    out["rolling_std_20"] = out[PRICE_COL].pct_change().rolling(20).std()
    obv_sign = np.sign(out[PRICE_COL].diff())
    obv_sign.iloc[0] = 0
    obv = (obv_sign * out["volume"]).cumsum()
    out["obv_roc"] = obv.pct_change(5)
    out[FEATURE_COLS] = out[FEATURE_COLS].replace([np.inf, -np.inf], np.nan)
    out = out.dropna(subset=FEATURE_COLS).reset_index(drop=True)
    out[TICKER_COL] = ticker
    validate_price_frame(out, ticker, "compute_technical_indicators")
    return out

def load_feature_frames(tickers):
    frames = {}
    rows = []
    for ticker in tickers:
        raw_df = read_one_minute_ticker(ticker)
        bars = resample_to_bars(raw_df, ticker)
        featured = compute_technical_indicators(bars, ticker)
        frames[ticker] = featured
        rows.append({"ticker": ticker, "raw_rows": len(raw_df), "bar_rows": len(bars), "feature_rows": len(featured), "start": featured[TIMESTAMP_COL].min(), "end": featured[TIMESTAMP_COL].max(), "warmup_rows_removed": len(bars) - len(featured), "feature_nan_count": int(featured[FEATURE_COLS].isna().sum().sum())})
    return frames, pd.DataFrame(rows)

if RUN_DATA_DIAGNOSTICS or RUN_TRAINING:
    feature_frames_by_ticker, feature_diagnostics_df = load_feature_frames(SELECTED_TICKERS)
else:
    feature_frames_by_ticker = {}
    feature_diagnostics_df = pd.DataFrame(columns=["ticker", "raw_rows", "bar_rows", "feature_rows", "start", "end", "warmup_rows_removed", "feature_nan_count"])

display(feature_diagnostics_df)

## `runner_utils` Label, Split, Scaler, And Window Pipeline

This section deliberately calls reviewed `runner_utils` functions for no-trade-band labels, chronological splits, train-only scaling, split-boundary invalid marking, and window creation. Running diagnostics can still read Drive data and build capped in-memory windows; the default is off.

In [ ]:
def dataset_labels(dataset):
    return np.asarray([int(dataset[index][1]) for index in range(len(dataset))], dtype=np.int64)

def dataset_window_summary(threshold_bps, ticker, split_name, dataset):
    y_values = dataset_labels(dataset)
    n_windows = int(len(y_values))
    n_up = int(np.sum(y_values == 1))
    n_down = int(np.sum(y_values == 0))
    return {"threshold_bps": threshold_bps, "ticker": ticker, "split": split_name, "n_valid_windows": n_windows, "n_up": n_up, "n_down": n_down, "up_pct": n_up / n_windows if n_windows else np.nan, "down_pct": n_down / n_windows if n_windows else np.nan, "minority_pct": min(n_up, n_down) / n_windows if n_windows else np.nan}

def label_frames_for_threshold(feature_frames, threshold_bps):
    labeled_frames = {}
    rows = []
    DataConfig(tickers=list(feature_frames), data_dir=DATA_ROOT, timestamp_col=TIMESTAMP_COL, price_col=PRICE_COL, label_mode="no_trade_band", threshold_bps=float(threshold_bps), feature_cols=FEATURE_COLS, train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO)
    for ticker, frame in feature_frames.items():
        labeled, diagnostics = make_no_trade_band_labels(frame, price_col=PRICE_COL, k=LABEL_HORIZON_K, threshold_bps=float(threshold_bps), timestamp_col=TIMESTAMP_COL)
        labeled[TICKER_COL] = ticker
        labeled_frames[ticker] = labeled
        n_total = diagnostics["n_total"]
        n_retained = diagnostics["n_up"] + diagnostics["n_down"]
        rows.append({"ticker": ticker, "threshold_bps": threshold_bps, **diagnostics, "n_retained": n_retained, "retained_pct": n_retained / n_total if n_total else np.nan})
    return labeled_frames, pd.DataFrame(rows)

def split_trim_scale_window(labeled_frames, threshold_bps):
    WindowConfig(window_size=WINDOW_SIZE, label_horizon_k=LABEL_HORIZON_K, stride=STRIDE)
    split_frames = {"train": [], "val": [], "test": []}
    for ticker, frame in labeled_frames.items():
        train_df, val_df, test_df = make_time_splits(frame, train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO, timestamp_col=TIMESTAMP_COL)
        for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
            trimmed = trim_labels_at_split_boundary(split_df, label_horizon_k=LABEL_HORIZON_K, label_col=LABEL_COL, timestamp_col=TIMESTAMP_COL, ticker_col=TICKER_COL)
            split_frames[split_name].append(trimmed)
    pooled_train = pd.concat(split_frames["train"], ignore_index=True)
    scaler = fit_scaler_on_train(pooled_train, feature_cols=FEATURE_COLS)
    transformed = {split_name: [transform_split(frame, scaler=scaler, feature_cols=FEATURE_COLS) for frame in frames] for split_name, frames in split_frames.items()}
    pooled_frames = {split_name: pd.concat(frames, ignore_index=True) for split_name, frames in transformed.items()}
    pooled_datasets = {split_name: WindowedClassificationDataset(frame, feature_cols=FEATURE_COLS, label_col=LABEL_COL, ticker_col=TICKER_COL, timestamp_col=TIMESTAMP_COL, window_size=WINDOW_SIZE, label_horizon_k=LABEL_HORIZON_K, stride=STRIDE) for split_name, frame in pooled_frames.items()}
    ticker_datasets = {"train": {}, "val": {}, "test": {}}
    for split_name, frames in transformed.items():
        for frame in frames:
            ticker = str(frame[TICKER_COL].iloc[0])
            ticker_datasets[split_name][ticker] = WindowedClassificationDataset(frame, feature_cols=FEATURE_COLS, label_col=LABEL_COL, ticker_col=TICKER_COL, timestamp_col=TIMESTAMP_COL, window_size=WINDOW_SIZE, label_horizon_k=LABEL_HORIZON_K, stride=STRIDE)
    rows = []
    for split_name, dataset in pooled_datasets.items():
        rows.append(dataset_window_summary(threshold_bps, "__pooled__", split_name, dataset))
    for split_name, by_ticker in ticker_datasets.items():
        for ticker, dataset in sorted(by_ticker.items()):
            rows.append(dataset_window_summary(threshold_bps, ticker, split_name, dataset))
    return {"threshold_bps": threshold_bps, "pooled_datasets": pooled_datasets, "ticker_datasets": ticker_datasets, "window_diagnostics": pd.DataFrame(rows)}

def prepare_thresholds(feature_frames, threshold_values):
    prepared = {}
    label_rows = []
    window_rows = []
    for threshold_bps in threshold_values:
        labeled_frames, label_df = label_frames_for_threshold(feature_frames, threshold_bps)
        prepared_item = split_trim_scale_window(labeled_frames, threshold_bps)
        prepared[threshold_bps] = prepared_item
        label_rows.append(label_df)
        window_rows.append(prepared_item["window_diagnostics"])
    return prepared, pd.concat(label_rows, ignore_index=True), pd.concat(window_rows, ignore_index=True)

thresholds_to_prepare = DIAGNOSTIC_THRESHOLD_BPS_LIST if RUN_DATA_DIAGNOSTICS else SELECTED_THRESHOLD_BPS_LIST if RUN_TRAINING else []
if thresholds_to_prepare:
    prepared_by_threshold, label_diagnostics_df, window_diagnostics_df = prepare_thresholds(feature_frames_by_ticker, thresholds_to_prepare)
else:
    prepared_by_threshold = {}
    label_diagnostics_df = pd.DataFrame()
    window_diagnostics_df = pd.DataFrame()

display(label_diagnostics_df)
display(window_diagnostics_df)

## `runner_utils` Metrics, Baselines, And Existing Models

The comparison rows below call `runner_utils.metrics` for dummy and always-class baselines. Neural smoke runs call existing `runner_utils` model classes and `runner_utils.trainer.train_one_epoch` / `evaluate`.

In [ ]:
def baseline_rows_for_eval(threshold_bps, train_dataset, eval_dataset, ticker):
    y_train = dataset_labels(train_dataset)
    y_eval = dataset_labels(eval_dataset)
    base_row = {"threshold_bps": threshold_bps, "ticker": ticker, "split": "test", "n_valid_windows": int(len(y_eval)), "n_up": int(np.sum(y_eval == 1)), "n_down": int(np.sum(y_eval == 0))}
    if len(y_train) == 0 or len(y_eval) == 0:
        return {**base_row, "dummy_stratified_macro_f1_mean": np.nan, "dummy_stratified_macro_f1_std": np.nan, "dummy_prior_macro_f1": np.nan, "always_up_macro_f1": np.nan, "always_down_macro_f1": np.nan}
    dummy_scores = []
    for dummy_seed in range(10):
        dummy_scores.append(dummy_baseline_metrics(y_train, y_eval, strategy="stratified", random_state=dummy_seed)["macro_f1"])
    dummy_prior = dummy_baseline_metrics(y_train, y_eval, strategy="prior", random_state=0)
    always_up = always_predict_baseline_metrics(y_eval, constant_label=1)
    always_down = always_predict_baseline_metrics(y_eval, constant_label=0)
    return {**base_row, "dummy_stratified_macro_f1_mean": float(np.mean(dummy_scores)), "dummy_stratified_macro_f1_std": float(np.std(dummy_scores)), "dummy_prior_macro_f1": float(dummy_prior["macro_f1"]), "always_up_macro_f1": float(always_up["macro_f1"]), "always_down_macro_f1": float(always_down["macro_f1"])}

def compute_baseline_table(prepared_by_threshold):
    rows = []
    for threshold_bps, prepared in prepared_by_threshold.items():
        train_dataset = prepared["pooled_datasets"]["train"]
        rows.append(baseline_rows_for_eval(threshold_bps, train_dataset, prepared["pooled_datasets"]["test"], "__pooled__"))
        for ticker, dataset in sorted(prepared["ticker_datasets"]["test"].items()):
            rows.append(baseline_rows_for_eval(threshold_bps, train_dataset, dataset, ticker))
    return pd.DataFrame(rows)

if prepared_by_threshold:
    dummy_baseline_summary_df = compute_baseline_table(prepared_by_threshold)
else:
    dummy_baseline_summary_df = pd.DataFrame()

display(dummy_baseline_summary_df)

In [ ]:
def build_baseline_model(model_name):
    if model_name == "lstm":
        return LSTMClassifier(input_size=len(FEATURE_COLS), hidden_size=64, num_layers=2, dropout=0.20)
    if model_name == "tcn":
        return TCNClassifier(input_size=len(FEATURE_COLS), num_channels=[64, 64, 64], kernel_size=3, dropout=0.20)
    if model_name == "dlinear":
        return DLinearClassifier(seq_len=WINDOW_SIZE, input_size=len(FEATURE_COLS), moving_avg_kernel=5)
    raise ValueError(f"model_name must be one of {AVAILABLE_BASELINE_MODELS}, got {model_name!r}")

proposed_architecture_spec_df = pd.DataFrame([
    {"component": "base", "choice": "DLinear-style seasonal/trend decomposition", "status": "future runner_utils implementation"},
    {"component": "multi_scale_moving_average", "choice": "[3, 6, 12, 24]", "status": "future runner_utils implementation"},
    {"component": "residual_tcn_branch", "choice": "dilations [1, 2, 4]", "status": "future runner_utils implementation"},
    {"component": "stock_embedding", "choice": "ticker id embedding for pooled training", "status": "requires dataset/model API design"},
    {"component": "comparison_policy", "choice": "must follow staged expansion and ablations before component claims", "status": "not trained in this notebook"},
])
display(proposed_architecture_spec_df)

## Guarded Training Entrypoint

Default training scope is one threshold, one ticker, one seed, one epoch, and LSTM only. Expanding threshold/model/ticker/seed axes should be done one reviewed rung at a time.

In [ ]:
def make_loader(dataset, shuffle, seed):
    generator = torch.Generator()
    generator.manual_seed(seed)
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, generator=generator)

def metric_row_from_runner_utils(metrics):
    return {"model_macro_f1": float(metrics["macro_f1"]), "model_balanced_accuracy": float(metrics["balanced_accuracy"]), "model_precision_macro": float(metrics["precision_macro"]), "model_recall_macro": float(metrics["recall_macro"]), "confusion_matrix": metrics["confusion_matrix"].tolist()}

def neural_result_row(threshold_bps, model_name, ticker, seed, metrics, eval_dataset, baseline_lookup, best_epoch, best_val_macro_f1, train_seconds):
    y_eval = dataset_labels(eval_dataset)
    baseline = baseline_lookup[(threshold_bps, ticker)]
    row = {"threshold_bps": threshold_bps, "model_name": model_name, "ticker": ticker, "seed": seed, "split": "test", "n_valid_windows": int(len(y_eval)), "n_up": int(np.sum(y_eval == 1)), "n_down": int(np.sum(y_eval == 0)), **metric_row_from_runner_utils(metrics), "dummy_stratified_macro_f1_mean": baseline["dummy_stratified_macro_f1_mean"], "dummy_stratified_macro_f1_std": baseline["dummy_stratified_macro_f1_std"], "dummy_prior_macro_f1": baseline["dummy_prior_macro_f1"], "always_up_macro_f1": baseline["always_up_macro_f1"], "always_down_macro_f1": baseline["always_down_macro_f1"], "best_epoch": best_epoch, "best_val_macro_f1": best_val_macro_f1, "train_seconds": train_seconds}
    row["delta_macro_f1_vs_dummy"] = row["model_macro_f1"] - row["dummy_stratified_macro_f1_mean"]
    return row

def run_one_neural_case(threshold_bps, prepared, model_name, seed, baseline_lookup):
    if model_name not in AVAILABLE_BASELINE_MODELS:
        raise ValueError(f"Notebook 04 only runs existing runner_utils baselines; got {model_name!r}")
    seed_everything(seed, deterministic=True)
    TrainConfig(batch_size=BATCH_SIZE, num_epochs=MAX_EPOCHS, learning_rate=LEARNING_RATE, early_stop_patience=1, device=device, seed=seed)
    model = build_baseline_model(model_name)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()
    train_loader = make_loader(prepared["pooled_datasets"]["train"], shuffle=True, seed=seed)
    val_loader = make_loader(prepared["pooled_datasets"]["val"], shuffle=False, seed=seed)
    best_state = None
    best_epoch = 0
    best_val_macro_f1 = -np.inf
    start_time = time.time()
    for epoch in range(1, MAX_EPOCHS + 1):
        train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, device=device, grad_clip=1.0)
        val_metrics, _, _ = evaluate(model, val_loader, criterion, device=device)
        if val_metrics["macro_f1"] > best_val_macro_f1:
            best_val_macro_f1 = float(val_metrics["macro_f1"])
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
        print(f"threshold={threshold_bps} model={model_name} seed={seed} epoch={epoch} train_loss={train_metrics['loss']:.6f} val_macro_f1={val_metrics['macro_f1']:.6f}")
    train_seconds = time.time() - start_time
    if best_state is not None:
        model.load_state_dict(best_state)
    rows = []
    test_loader = make_loader(prepared["pooled_datasets"]["test"], shuffle=False, seed=seed)
    pooled_metrics, _, _ = evaluate(model, test_loader, criterion, device=device)
    rows.append(neural_result_row(threshold_bps, model_name, "__pooled__", seed, pooled_metrics, prepared["pooled_datasets"]["test"], baseline_lookup, best_epoch, best_val_macro_f1, train_seconds))
    for ticker, dataset in sorted(prepared["ticker_datasets"]["test"].items()):
        ticker_loader = make_loader(dataset, shuffle=False, seed=seed)
        ticker_metrics, _, _ = evaluate(model, ticker_loader, criterion, device=device)
        rows.append(neural_result_row(threshold_bps, model_name, ticker, seed, ticker_metrics, dataset, baseline_lookup, best_epoch, best_val_macro_f1, train_seconds))
    return rows

def run_model_comparison():
    if not RUN_TRAINING:
        print("RUN_TRAINING=False; no neural training was run.")
        return pd.DataFrame()
    baseline_lookup = {(row["threshold_bps"], row["ticker"]): row for row in dummy_baseline_summary_df.to_dict(orient="records")}
    rows = []
    for threshold_bps in SELECTED_THRESHOLD_BPS_LIST:
        prepared = prepared_by_threshold[threshold_bps]
        for model_name in SELECTED_MODELS:
            for seed in SELECTED_SEEDS:
                rows.extend(run_one_neural_case(threshold_bps, prepared, model_name, seed, baseline_lookup))
    return pd.DataFrame(rows)

per_ticker_results_df = run_model_comparison()
if len(per_ticker_results_df):
    pooled_summary_df = per_ticker_results_df[per_ticker_results_df["ticker"] == "__pooled__"].reset_index(drop=True)
    threshold_model_summary_df = per_ticker_results_df[per_ticker_results_df["ticker"] != "__pooled__"].groupby(["threshold_bps", "model_name", "seed"], as_index=False).agg(model_macro_f1_mean=("model_macro_f1", "mean"), delta_macro_f1_vs_dummy_mean=("delta_macro_f1_vs_dummy", "mean"), positive_delta_tickers=("delta_macro_f1_vs_dummy", lambda values: int((values > 0).sum())), n_tickers=("ticker", "nunique"))
else:
    pooled_summary_df = pd.DataFrame()
    threshold_model_summary_df = planned_grid_df.copy()
    threshold_model_summary_df["status"] = "planned_not_run"

display(per_ticker_results_df)
display(pooled_summary_df)
display(threshold_model_summary_df)

## Optional Artifact Writing

Artifact writing is disabled by default and should only be enabled for a separately reviewed artifact-policy smoke. This cell writes only summary tables created by this notebook.

In [ ]:
def write_artifacts_if_enabled():
    if not WRITE_ARTIFACTS:
        return pd.DataFrame([{"write_artifacts": False, "artifact_root": ARTIFACT_ROOT, "status": "skipped"}])
    artifact_dir = Path(ARTIFACT_ROOT)
    if artifact_dir.exists() and not ALLOW_OVERWRITE:
        raise FileExistsError(f"Artifact directory already exists and ALLOW_OVERWRITE=False: {artifact_dir}")
    artifact_dir.mkdir(parents=True, exist_ok=True)
    feature_diagnostics_df.to_csv(artifact_dir / "feature_diagnostics.csv", index=False)
    label_diagnostics_df.to_csv(artifact_dir / "label_diagnostics.csv", index=False)
    window_diagnostics_df.to_csv(artifact_dir / "window_diagnostics.csv", index=False)
    dummy_baseline_summary_df.to_csv(artifact_dir / "dummy_baseline_summary.csv", index=False)
    per_ticker_results_df.to_csv(artifact_dir / "per_ticker_results.csv", index=False)
    pooled_summary_df.to_csv(artifact_dir / "pooled_summary.csv", index=False)
    threshold_model_summary_df.to_csv(artifact_dir / "threshold_model_summary.csv", index=False)
    manifest = {"notebook": "04_ian_no_trade_band_multiscale_dlinear_tcn.ipynb", "full_run": FULL_RUN, "run_data_diagnostics": RUN_DATA_DIAGNOSTICS, "run_training": RUN_TRAINING, "write_artifacts": WRITE_ARTIFACTS, "allow_overwrite": ALLOW_OVERWRITE, "selected_tickers": SELECTED_TICKERS, "selected_threshold_bps_list": SELECTED_THRESHOLD_BPS_LIST, "diagnostic_threshold_bps_list": DIAGNOSTIC_THRESHOLD_BPS_LIST, "selected_models": SELECTED_MODELS, "selected_seeds": SELECTED_SEEDS, "max_raw_rows_per_ticker": MAX_RAW_ROWS_PER_TICKER, "max_epochs": MAX_EPOCHS, "selection_bias_disclosure": "No-trade-band metrics estimate conditional direction classification on retained samples, not full-market direction prediction."}
    (artifact_dir / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return pd.DataFrame([{"write_artifacts": True, "artifact_root": str(artifact_dir), "status": "written"}])

artifact_status_df = write_artifacts_if_enabled()
display(artifact_status_df)

## Interpretation Guardrails And Next Rungs

- Running all cells with defaults does not mount Drive, load data, train, or write artifacts.
- `RUN_DATA_DIAGNOSTICS=True` reads capped Drive data and computes feature, label, scaler, baseline, and window diagnostics.
- `RUN_TRAINING=True` should stay on one reviewed axis at a time; default is threshold `5`, ticker `CSCO`, seed `42`, model `lstm`, one epoch, no artifacts.
- `threshold_bps=10` is sensitivity-only, not a main setting.
- `threshold_bps=0` is a no-band reference, not a no-trade-band main setting.
- The proposed stock-aware multi-scale DLinear TCN requires a separate `runner_utils` implementation/review path before comparison.
- Do not claim model signal, robustness, or component causality from smoke-scale runs.